# 03. 터널 입구 사진 세기 (Mapillary)

**목적**: 모델③ 정답 데이터(사람이 붙이는 "입구 안이 얼마나 안 보이나" 상/중/하)를 만들 사진이 충분한지 먼저 확인해요.

**무엇을 세나**
- 고속도로 터널(`tunnel_expressway.csv`)의 양쪽 입구마다, 입구 앞 20~150 m에서 **입구를 바라보고** 찍은 Mapillary 사진
- 낮(07~19시 KST) 사진만, 360° 사진 제외
- 사진마다 찍은 시각·위치·카메라 방향이 있어서 해 높이·해와 터널 방향 각도를 계산할 수 있어요 (1팀과 같은 이유)

**준비**
- `sunsight_data/.env`에 `MAPILLARY_TOKEN=...` 한 줄 추가 (mapillary.com → 개발자 → 앱 등록 → Client Token, 무료). 1팀 이성진 님이 이미 쓰는 토큰이 있으면 그걸 써도 돼요. 공유 금지.
- 실행 위치: `머신러닝3조/sunsight_video` (이 노트북이 있는 폴더)

**결과**: `data/mapillary_tunnel_candidates.csv`, 개수 표. 사진 다운로드는 마지막 셀(선택)에서만 해요.

In [9]:
import os, math, time, json
from pathlib import Path
import numpy as np, pandas as pd, requests

ROOT = Path('..') / 'sunsight_data'
TUNNEL_CSV = ROOT / 'data' / '2_중간' / 'tunnel_expressway.csv'
OUT = Path('data'); OUT.mkdir(exist_ok=True)

def read_env(p):
    env = {}
    if p.exists():
        for line in p.read_text(encoding='utf-8').splitlines():
            if '=' in line and not line.strip().startswith('#'):
                k, v = line.split('=', 1); env[k.strip()] = v.strip().strip('"')
    return env
TOKEN = read_env(ROOT / '.env').get('MAPILLARY_TOKEN') or os.environ.get('MAPILLARY_TOKEN')
assert TOKEN, 'sunsight_data/.env 에 MAPILLARY_TOKEN=... 을 넣어 주세요'

tun = pd.read_csv(TUNNEL_CSV, encoding='utf-8-sig')
print(len(tun), '개 터널'); print(list(tun.columns))

1327 개 터널
['tunnel_id', '터널명', '도로노선명', '도로노선번호', '도로노선방향', 'start_lat', 'start_lon', 'end_lat', 'end_lon', 'length_m', '터널준공연도', '관리기관명']


## 1. 입구 좌표 열 고르기
위에 출력된 열 이름을 보고, 시작점·종료점 위도/경도 열을 아래 4개에 적어 주세요. 자동으로 추측한 값이 맞으면 그대로 두면 돼요.

In [10]:
def guess(words):
    for c in tun.columns:
        if all(w in c for w in words): return c
    return None
START_LAT = guess(['start_lat']) or guess(['시작', '위도']); START_LON = guess(['start_lon']) or guess(['시작', '경도'])
END_LAT   = guess(['end_lat']) or guess(['종료', '위도']); END_LON   = guess(['end_lon']) or guess(['종료', '경도'])
NAME      = guess(['터널명']) or tun.columns[0]
print('시작', START_LAT, START_LON, '| 종료', END_LAT, END_LON, '| 이름', NAME)
assert all([START_LAT, START_LON, END_LAT, END_LON]), '열 이름을 직접 적어 주세요'

# 입구 = 터널 양 끝점. 각 끝점은 한쪽 방향 차량의 '진입' 입구
portals = []
for i, r in tun.iterrows():
    for side, la, lo in [('시작', START_LAT, START_LON), ('종료', END_LAT, END_LON)]:
        if pd.notna(r[la]) and pd.notna(r[lo]):
            portals.append({'tunnel_idx': i, 'tunnel': r[NAME], 'side': side, 'lat': float(r[la]), 'lon': float(r[lo])})
portals = pd.DataFrame(portals)
print(len(portals), '개 입구')

시작 start_lat start_lon | 종료 end_lat end_lon | 이름 터널명
2654 개 입구


## 2. 입구 주변 사진 받아 오기 (목록만, 사진 파일은 아님)
입구 주변 약 ±170 m 상자 안의 사진 목록을 받아요. 입구 2,600여 곳이라 10~20분 걸릴 수 있어요. 중간에 끊겨도 `mapillary_raw.jsonl`에 쌓인 곳은 다시 받지 않아요.

In [11]:
FIELDS = 'id,captured_at,compass_angle,computed_compass_angle,geometry,computed_geometry,is_pano,thumb_1024_url'
D = 0.0015   # 위도 0.0015° ≈ 167 m
RAW = OUT / 'mapillary_raw.jsonl'
done = set()
if RAW.exists():
    for line in RAW.open(encoding='utf-8'):
        done.add(json.loads(line)['portal'])

with RAW.open('a', encoding='utf-8') as f:
    for k, p in portals.iterrows():
        if k in done: continue
        dlon = D / math.cos(math.radians(p.lat))
        bbox = f'{p.lon-dlon},{p.lat-D},{p.lon+dlon},{p.lat+D}'
        for attempt in range(3):
            try:
                r = requests.get('https://graph.mapillary.com/images',
                                 params={'access_token': TOKEN, 'bbox': bbox, 'fields': FIELDS, 'limit': 2000}, timeout=30)
                r.raise_for_status(); data = r.json().get('data', []); break
            except Exception as e:
                print(k, '재시도', e); time.sleep(2 ** attempt); data = None
        if data is None: continue
        f.write(json.dumps({'portal': int(k), 'images': data}, ensure_ascii=False) + '\n')
        if k % 200 == 0: print(k, '/', len(portals))
        time.sleep(0.05)
print('완료')

완료


## 3. 입구를 바라보는 낮 사진만 남기기
- 입구까지 거리 20~150 m (너무 가까우면 이미 터널 안, 멀면 입구가 작게 보임)
- 카메라 방향과 '사진 → 입구' 방향 차이 30° 이하 (입구를 정면으로 봄)
- 낮 07~19시 KST, 360° 사진 제외

기준값(20·150 m, 30°)은 사진을 몇 장 열어 보고 형이 조정해요. 지금은 임시값이에요.

In [12]:
def bearing(lat1, lon1, lat2, lon2):
    p1, p2 = math.radians(lat1), math.radians(lat2); dl = math.radians(lon2 - lon1)
    x = math.sin(dl) * math.cos(p2); y = math.cos(p1) * math.sin(p2) - math.sin(p1) * math.cos(p2) * math.cos(dl)
    return (math.degrees(math.atan2(x, y)) + 360) % 360
def dist_m(lat1, lon1, lat2, lon2):
    return 6371000 * math.hypot(math.radians(lat2 - lat1), math.radians(lon2 - lon1) * math.cos(math.radians(lat1)))

rows = []
for line in RAW.open(encoding='utf-8'):
    rec = json.loads(line); p = portals.loc[rec['portal']]
    for im in rec['images']:
        g = (im.get('computed_geometry') or im.get('geometry') or {}).get('coordinates')
        head = im.get('computed_compass_angle', im.get('compass_angle'))
        if not g or head is None or im.get('is_pano'): continue
        lon, lat = g
        d = dist_m(lat, lon, p.lat, p.lon)
        diff = abs((bearing(lat, lon, p.lat, p.lon) - head + 180) % 360 - 180)
        rows.append({'portal': rec['portal'], 'tunnel': p.tunnel, 'side': p.side, 'image_id': im['id'],
                     'captured_at': pd.to_datetime(im['captured_at'], unit='ms', utc=True).tz_convert('Asia/Seoul'),
                     'lat': lat, 'lon': lon, 'heading': head, 'dist_m': round(d, 1), 'angle_diff': round(diff, 1),
                     'thumb': im.get('thumb_1024_url')})
img = pd.DataFrame(rows)
print('입구 주변 전체 사진', len(img))
img['hour'] = img.captured_at.dt.hour
cand = img[img.dist_m.between(20, 150) & (img.angle_diff <= 30) & img.hour.between(7, 18)].copy()
cand['month'] = cand.captured_at.dt.month; cand['year'] = cand.captured_at.dt.year
cand.drop(columns='thumb').to_csv(OUT / 'mapillary_tunnel_candidates.csv', index=False, encoding='utf-8-sig')
cand[['image_id', 'thumb']].to_csv(OUT / 'mapillary_thumb_urls.csv', index=False)   # 링크는 시간이 지나면 만료됨
print('후보 사진', len(cand), '| 입구', cand.portal.nunique(), '/', len(portals), '| 터널', cand.tunnel.nunique())

입구 주변 전체 사진 12627
후보 사진 2644 | 입구 315 / 2654 | 터널 169


## 4. 형에게 보고할 표
라벨에 쓰려면 **입구 수**와 **시각·계절이 고르게 퍼졌는지**가 중요해요. 같은 입구 사진은 대부분 같은 날 연속 촬영이라, 실제로 쓸 수 있는 건 '입구 × 촬영 날' 단위예요.

In [13]:
cand['date'] = cand.captured_at.dt.date
uniq = cand.groupby(['portal', 'date']).size()
print('입구 × 촬영 날 조합:', len(uniq))
print('\n시간대별 (입구×날):'); print(cand.drop_duplicates(['portal', 'date', 'hour']).hour.value_counts().sort_index().to_string())
print('\n월별 (입구×날):'); print(cand.drop_duplicates(['portal', 'date']).month.value_counts().sort_index().to_string())
print('\n연도별:'); print(cand.drop_duplicates(['portal', 'date']).year.value_counts().sort_index().to_string())
print('\n사진 많은 입구 상위 10:'); print(cand.groupby(['tunnel', 'side']).date.nunique().sort_values(ascending=False).head(10).to_string())

입구 × 촬영 날 조합: 353

시간대별 (입구×날):
hour
8     13
9     28
10    86
11    15
12    25
13    28
14    62
15    33
16    26
17    41
18    22

월별 (입구×날):
month
2       2
3       8
4      15
5     150
6      19
9      30
10     81
12     48

연도별:
year
2016      2
2017      9
2019     42
2023    165
2024     93
2025     42

사진 많은 입구 상위 10:
tunnel     side
구리터널(포천)   종료      2
청라2지하차도    시작      2
청라3지하차도    종료      2
중원터널(양평)   종료      2
중원터널(창원)   시작      2
           종료      2
월문3터널(양양)  종료      2
이천터널(서울)   종료      2
이천터널(양양)   시작      2
중랑IC1지하차도  시작      2


## 5. (선택) 라벨용 사진 내려받기
입구 × 촬영 날마다 입구에서 가장 가까운 1장만 받아요. 위 표를 형이 보고 "라벨할 만하다" 판단한 뒤에 실행하세요.

In [14]:
DOWNLOAD = True   # True로 바꾸면 실행
if DOWNLOAD:
    IMG_DIR = OUT / 'tunnel_photos'; IMG_DIR.mkdir(exist_ok=True)
    pick = cand.sort_values('dist_m').drop_duplicates(['portal', 'date'])
    for _, r in pick.iterrows():
        fp = IMG_DIR / f"{r.image_id}.jpg"
        if fp.exists() or not r.thumb: continue
        try: fp.write_bytes(requests.get(r.thumb, timeout=30).content)
        except Exception as e: print(r.image_id, e)
    pick.drop(columns='thumb').to_csv(OUT / 'tunnel_photos_meta.csv', index=False, encoding='utf-8-sig')
    print(len(pick), '장 →', IMG_DIR)

353 장 → data\tunnel_photos
